# 🧬 Regime Lab – Quant Researcher
**Workflow**: Load returns → GMM Regime Detection → Regime-conditional stats → Rotation signal simulation

---

In [ ]:
from qtrader.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

session = AnalystSession(role=RoleContext.RESEARCHER)

SYMBOL = 'BTC-USD'
TIMEFRAME = '1d'
N_REGIMES = 3  # Bull / Bear / Sideways

## 1. Load & Prepare Features

In [ ]:
try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME)
except Exception:
    df = session.sample_ohlcv(symbol='BTC', days=500)

df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 21])
df = df.drop_nulls()

# Feature set for GMM
regime_features = ['returns', 'vol_21', 'rsi_14']
X = df.select(regime_features).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Regime input shape: {X_scaled.shape}")

## 2. GMM Regime Detection

In [ ]:
gmm = GaussianMixture(n_components=N_REGIMES, covariance_type='full', random_state=42, n_init=5)
gmm.fit(X_scaled)
labels = gmm.predict(X_scaled)
probs  = gmm.predict_proba(X_scaled)

df = df.with_columns(pl.Series('regime', labels))
print(f"BIC: {gmm.bic(X_scaled):.1f} | AIC: {gmm.aic(X_scaled):.1f}")
print(f"Regime counts: {dict(zip(*np.unique(labels, return_counts=True)))}")

## 3. Regime-Conditional Statistics

In [ ]:
cond_stats = (
    df.group_by('regime')
    .agg([
        pl.col('returns').mean().alias('mean_return'),
        pl.col('returns').std().alias('vol'),
        pl.col('returns').count().alias('count'),
        pl.col('rsi_14').mean().alias('mean_rsi'),
    ])
    .sort('regime')
)
# Add Sharpe per regime
cond_stats = cond_stats.with_columns(
    (pl.col('mean_return') / (pl.col('vol') + 1e-10) * (252**0.5)).alias('regime_sharpe')
)
cond_stats

## 4. Regime Timeline Visualisation

In [ ]:
palette = ['#34d399', '#f87171', '#a78bfa']
close = df['close'].to_numpy()
regs  = df['regime'].to_numpy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), facecolor='#0f1117', sharex=True)
for ax in [ax1, ax2]:
    ax.set_facecolor('#0f1117')
    for sp in ax.spines.values(): sp.set_color('#334155')
    ax.tick_params(colors='#94a3b8')
    ax.grid(color='#1e293b', linestyle='--', linewidth=0.5)

ax1.plot(close, color='#e2e8f0', linewidth=0.8)
for r in range(N_REGIMES):
    mask = regs == r
    ax1.fill_between(range(len(close)), close.min(), close, where=mask, alpha=0.2, color=palette[r], label=f'Regime {r}')
ax1.set_ylabel('Close Price', color='#94a3b8')
ax1.set_title(f'{SYMBOL} – GMM Regimes ({N_REGIMES})', color='#e2e8f0')
ax1.legend(facecolor='#1e293b', labelcolor='#e2e8f0')

ax2.plot(probs[:, 0], color=palette[0], linewidth=1, label='P(Regime 0)')
ax2.plot(probs[:, 1], color=palette[1], linewidth=1, label='P(Regime 1)')
if N_REGIMES > 2:
    ax2.plot(probs[:, 2], color=palette[2], linewidth=1, label='P(Regime 2)')
ax2.set_ylabel('Posterior Prob.', color='#94a3b8')
ax2.legend(facecolor='#1e293b', labelcolor='#e2e8f0')
plt.tight_layout()
plt.show()

## 5. Rotation Signal (Bull-only)

Simulate strategy: go long only in the highest-Sharpe regime, cash otherwise.

In [ ]:
# Identify best regime by Sharpe
best_regime = int(cond_stats.sort('regime_sharpe', descending=True).head(1)['regime'].item())
print(f"Best regime: {best_regime}")

df = df.with_columns(
    pl.when(pl.col('regime') == best_regime).then(1).otherwise(0).alias('regime_signal')
)
bt = session.run_vector_backtest(df, signal_col='regime_signal')
m = session.compute_extended_metrics(bt['equity_curve'])
print({k: round(v, 4) for k, v in m.items()})